# Week 1 — Day 5: Multi-Step LLM Business Solution

## Business Challenge

Build a small AI application that produces a company brochure for:

- prospective customers
- investors
- potential employees

The application will use multiple LLM calls rather than a single prompt.

## Goals

- Understand multi-step LLM workflows
- Separate tasks into individual functions
- Use one LLM call to produce structured information
- Use another LLM call for content generation
- Understand how outputs from one step become inputs to another
- Learn streaming responses
- Use the modern OpenAI Responses API
- Compare the course implementation with current OpenAI APIs

## Architecture

Company Information
        ↓
Information Selection
        ↓
Structured Data
        ↓
Brochure Prompt
        ↓
OpenAI Responses API
        ↓
Markdown Brochure

## Official Documentation

OpenAI Responses API:
https://platform.openai.com/docs/api-reference/responses

OpenAI Structured Outputs:
https://platform.openai.com/docs/guides/structured-outputs

OpenAI Streaming:
https://platform.openai.com/docs/guides/streaming-responses

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv("../.env", override=True)

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [5]:
company_name = "SanvyAI"

company_information = """
SanvyAI is an AI technology company focused on building production-grade
Generative AI and Agentic AI solutions.

The company develops systems involving:
- Retrieval-Augmented Generation
- AI agents
- document intelligence
- enterprise search
- workflow automation

Its target customers include startups and businesses that want to integrate
AI into their existing products and internal workflows.

The engineering stack includes Python, FastAPI, LangChain, LangGraph,
PostgreSQL, Redis, Docker, Kubernetes and AWS.

The company values secure, reliable and scalable AI systems.
"""

https://platform.openai.com/docs/guides/structured-outputs
https://platform.openai.com/docs/api-reference/responses

## Step 1 — Extract Structured Company Information

Instead of asking the model to return arbitrary JSON, we define the exact structure
our application expects.

This gives us predictable fields that can be passed safely into the next step.

In [6]:
from pydantic import BaseModel

In [7]:
class CompanyProfile(BaseModel):
    name: str
    summary: str
    target_customers: list[str]
    products_services: list[str]
    technologies: list[str]
    company_values: list[str]

In [8]:
profile_response = client.responses.parse(
    model="gpt-5-mini",
    instructions="""
    You extract useful company information for creating a professional brochure.

    Only use information provided by the user.
    Do not invent facts.
    """,
    input=company_information,
    text_format=CompanyProfile
)

In [9]:
company_profile = profile_response.output_parsed

In [15]:
company_profile



CompanyProfile(name='SanvyAI', summary='SanvyAI is an AI technology company focused on building production-grade Generative AI and Agentic AI solutions. The company develops systems involving Retrieval-Augmented Generation, AI agents, document intelligence, enterprise search, and workflow automation.', target_customers=['Startups', 'Businesses that want to integrate AI into their existing products and internal workflows'], products_services=['Retrieval-Augmented Generation systems', 'AI agents', 'Document intelligence', 'Enterprise search', 'Workflow automation', 'Integration support for embedding AI into existing products and internal workflows'], technologies=['Python', 'FastAPI', 'LangChain', 'LangGraph', 'PostgreSQL', 'Redis', 'Docker', 'Kubernetes', 'AWS'], company_values=['Secure AI systems', 'Reliable AI systems', 'Scalable AI systems'])

In [16]:
company_profile.name

'SanvyAI'

In [17]:
company_profile.target_customers

['Startups',
 'Businesses that want to integrate AI into their existing products and internal workflows']

Step 2: use the structured CompanyProfile to generate the brochure.

https://developers.openai.com/api/docs/guides/text
https://developers.openai.com/api/docs/guides/structured-outputs
https://developers.openai.com/api/docs/guides/streaming-responses

## Step 2 — Generate the Brochure

The first LLM call converted unstructured company information into a validated
`CompanyProfile`.

Now the second LLM call will use that structured data to create a professional
brochure.

### Workflow

Raw company text  
→ Structured extraction  
→ `CompanyProfile`  
→ Brochure generation  
→ Markdown output

In [18]:
profile_json = company_profile.model_dump_json(indent=2)

print(profile_json)

{
  "name": "SanvyAI",
  "summary": "SanvyAI is an AI technology company focused on building production-grade Generative AI and Agentic AI solutions. The company develops systems involving Retrieval-Augmented Generation, AI agents, document intelligence, enterprise search, and workflow automation.",
  "target_customers": [
    "Startups",
    "Businesses that want to integrate AI into their existing products and internal workflows"
  ],
  "products_services": [
    "Retrieval-Augmented Generation systems",
    "AI agents",
    "Document intelligence",
    "Enterprise search",
    "Workflow automation",
    "Integration support for embedding AI into existing products and internal workflows"
  ],
  "technologies": [
    "Python",
    "FastAPI",
    "LangChain",
    "LangGraph",
    "PostgreSQL",
    "Redis",
    "Docker",
    "Kubernetes",
    "AWS"
  ],
  "company_values": [
    "Secure AI systems",
    "Reliable AI systems",
    "Scalable AI systems"
  ]
}


In [20]:
def generate_brochure(profile: CompanyProfile) -> str:

    profile_json = profile.model_dump_json(indent=2)

    response = client.responses.create(
        model="gpt-5-mini",

        instructions="""
        You are a professional business content writer.

        Create a concise and professional company brochure.

        The brochure should contain:

        # Company Name
        ## About
        ## Who We Help
        ## Products & Services
        ## Technology
        ## Company Values

        Use only the provided company information.
        Do not invent facts.

        Format the response using Markdown.
        """,

        input=profile_json
    )

    return response.output_text

In [21]:
brochure = generate_brochure(company_profile)

In [22]:
display(Markdown(brochure))

# SanvyAI

## About
SanvyAI is an AI technology company focused on building production-grade Generative AI and Agentic AI solutions. The company develops systems involving Retrieval-Augmented Generation, AI agents, document intelligence, enterprise search, and workflow automation.

## Who We Help
- Startups  
- Businesses that want to integrate AI into their existing products and internal workflows

## Products & Services
- Retrieval-Augmented Generation systems  
- AI agents  
- Document intelligence  
- Enterprise search  
- Workflow automation  
- Integration support for embedding AI into existing products and internal workflows

## Technology
- Python  
- FastAPI  
- LangChain  
- LangGraph  
- PostgreSQL  
- Redis  
- Docker  
- Kubernetes  
- AWS

## Company Values
- Secure AI systems  
- Reliable AI systems  
- Scalable AI systems